<a href="https://colab.research.google.com/github/bvvnmanikanta/Scaler-AI-Notebooks/blob/main/class04_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Class 4 · LLM Built-in Power Tools — class notebook

*AI & Agents: From Talking to AI to Building One · Scaler · Rishav Singh · v7, 13 Nov 2026*

**First: File → Save a copy in Drive.** Then this copy is yours to edit.

**Today's rung:** Classes 2–3 taught you to *write* prompts. Today you *package* them: as skills, Projects and lab settings a whole team can reuse. Industry calls this **context engineering**.

**How to use it.** When a slide says **Live demo**, **Hands-on** or **Run along** with *Notebook · Section N*, come to that section here.
- **To copy a prompt:** click inside its code cell, press **Ctrl+A** then **Ctrl+C** (Cmd on a Mac). Or run the cell and copy the printed text.
- Cells that start with `%%writefile` **create a file** when you run them; the cell after it downloads the file.
- No API key needed. Two optional cells (Sections 6 and 7) use a free Gemini API key if you have one, and skip cleanly if you don't.

**Skills or Gems?** On **17 Nov 2026** Google turned Gems into **skills** for personal Google accounts (Gems become read-only, then go away on 17 Dec). Same parts: name, description, instructions, files. Each section gives the skill path first and the Gem path second. Skills need age 18+.

## Section 0 · Tabs to open
1. **gemini.google.com**, signed in with Google (Sections 1–4 and mini build A)
2. **claude.ai**, a free account is enough (Section 5 and mini build B). Free plan: up to 5 projects.
3. **aistudio.google.com**, signed in (Sections 6–7)
4. This notebook

## Section 1 · Live demo: build the Code Review Bot as a skill *(slide 10 · watch; you build one in Section 3)*

**Before Rishav types the description:** write your own one-line description in Zoom chat. *When should Gemini pick this skill by itself?* Then compare it with `SKILL_DESCRIPTION` below. The description is the only part of a skill the model always sees, so write it like you'd name an API endpoint: specific.

**Skill path (17 Nov on)**
1. gemini.google.com → **Customize** (if you don't see it, click **More** first) → **Skills** → **Create** → **Create manually**.
2. Name `SKILL_NAME` (all lowercase, hyphens), description `SKILL_DESCRIPTION`, instructions `INSTRUCTIONS`.
3. **Add** → images or files → `banned_patterns.md` (made by the cells below) → **Upload** → **Create**.
4. Use it in any chat: type **/** (or **@**) and its name, or just ask and let Gemini pick it.

**Gem path (before 17 Nov)**
1. gemini.google.com → **Gems** (first time: **Settings and help → Gems**) → **New Gem**.
2. Name `GEM_NAME`, description `GEM_DESCRIPTION`, instructions `INSTRUCTIONS`.
3. **Knowledge → Add files → Upload files** → `banned_patterns.md` → try it in the preview → **Save**.

In [ ]:
SKILL_NAME = "review-python-code"
SKILL_DESCRIPTION = ("Reviews Python code for bugs, security issues and PEP 8 style, rates readability and "
                     "suggests one refactor. Use when the user shares Python code or asks for a code review.")

GEM_NAME = "Code Review Bot"
GEM_DESCRIPTION = "A strict code reviewer that checks Python code for bugs, security issues, and style violations."

INSTRUCTIONS = """You are a senior Python code reviewer at a top-tier tech company.

Your Review Process
For every code snippet shared, you must:
1. Check for bugs and logical errors
2. Flag security vulnerabilities (SQL injection, hardcoded secrets,
   unsafe deserialization, etc.)
3. Note PEP 8 style violations
4. Rate readability on a scale of 1-5
5. Suggest the single most impactful refactor

Output Format
Use this structure for every review:
- Bugs: (list or "None found")
- Security: (list or "No issues")
- Style: (top 3 violations or "Clean")
- Readability: X/5 with one-line justification
- Top Refactor: (one specific suggestion with before/after code)

Constraints
- Keep reviews under 300 words
- Be direct and constructive — no fluff
- If the code is good, say so briefly and move on
- Always suggest type hints if they're missing"""

print(SKILL_DESCRIPTION, INSTRUCTIONS, sep="\n\n")

**Knowledge file.** Run the next cell to create `banned_patterns.md`, then the one after it to download it.

In [ ]:
%%writefile banned_patterns.md
# Banned patterns (payments team)
- Bare `except:`. Always catch a specific exception.
- `print()` in service code. Use the logging module.
- `eval()` or `exec()` on any input.
- Hardcoded secrets, API keys or passwords. Read them from environment variables.
- Mutable default arguments, like `def add(item, items=[])`.
- Unused imports and unused variables.

In [ ]:
try:
    from google.colab import files
    files.download("banned_patterns.md")
except ImportError:
    print("Not in Colab: banned_patterns.md is saved next to this notebook.")

## Section 2 · Live demo: you vs the skill *(slide 12 · 60-second race)*

**Your job first (60 s):** read `CODE_TO_REVIEW` and type **three problems** in Zoom chat before the skill answers.

**Run 1:** new chat → type `/review-python-code` → paste `CODE_TO_REVIEW`. *(Gem path: open the Code Review Bot from the sidebar and paste.)*

**Run 2 (skill path):** a **new** chat, **no "/"**: send `AUTO_PICK_MESSAGE` followed by the script. Did Gemini pick the skill by itself? If not, tighten the description and try again.

**Watch for:** every line of the review comes from a rule in `INSTRUCTIONS` or `banned_patterns.md`. Which rule?

In [ ]:
AUTO_PICK_MESSAGE = "Can you review this Python before I open a PR?"

CODE_TO_REVIEW = """import random
import math as m
import datetime

def calcAverage(numbers):
    total = 0
    for i in numbers:
        total += i
    return total / len(numbers)

def generate_numbers(n):
    nums = []
    for i in range(0, n):
        nums.append(random.randint(1, 100))
    return nums

def printReport(data):
    avg = calcAverage(data)
    print("Average is:", avg)
    print("Max value is:", max(data))
    print("Min value is:", min(data))

def weird_function(x,y):
    if x > y:
        return x - y
    else:
        return y - x

def main():
    count = 10
    numbers = generate_numbers(count)

    printReport(numbers)

    # inconsistent naming + unused variable
    currentTime = datetime.datetime.now()
    unused_var = "hello"

    for i in range(len(numbers)):
        if numbers[i] % 2 == 0:
            print("Even:", numbers[i])
        else:
            print("Odd:", numbers[i])

    # inconsistent spacing and logic
    result=weird_function(5 ,10)
    print("Result:",result)

    # unnecessary complexity
    squares = list(map(lambda x: x**2, numbers))
    print("Squares:", squares)

    # possible division by zero (edge case ignored)
    empty_list = []
    try:
        print(calcAverage(empty_list))
    except:
        print("Error calculating average")

if __name__ == "__main__":
    main()
"""

print(AUTO_PICK_MESSAGE, CODE_TO_REVIEW, sep="\n\n")

**After the review:** run the script. Look at the last line it prints, then at the `except:` again. What did it hide?

In [ ]:
exec(CODE_TO_REVIEW, {"__name__": "__main__"})

## Section 3 · Hands-on: your Docs Drafter *(slide 14 · 10 min)*

A **docstring** is the description in triple quotes at the top of a function. Google style: a summary line, then **Args**, **Returns**, **Raises** and **Example**.

**Skill path:** Customize → Skills → **Create** → **Create manually** → name `SKILL_NAME`, description `SKILL_DESCRIPTION`, instructions `INSTRUCTIONS` → **Create** → new chat → `/write-python-docstrings` + `TEST_CODE_1`. Then a **new** chat with `TEST_CODE_2` and **no "/"**: does Gemini pick it?

**Gem path:** Gems → **New Gem** → name `GEM_NAME`, description `GEM_DESCRIPTION`, paste `INSTRUCTIONS` → preview with `TEST_CODE_1` → **Save** → reopen it from the sidebar with `TEST_CODE_2`.

**Watch for:** only the docstring comes back. No "Sure! Here's your docstring…".
**Post in Zoom chat:** ✅ only the docstring, or ❌ + the first line it wrote.

In [ ]:
SKILL_NAME = "write-python-docstrings"
SKILL_DESCRIPTION = ("Writes complete Google-style docstrings for Python functions and classes. "
                     "Use when the user shares Python code and wants it documented.")

GEM_NAME = "Docs Drafter"
GEM_DESCRIPTION = "Writes Google-style Python docstrings for any function or class."

INSTRUCTIONS = """You are a documentation specialist who writes Google-style Python docstrings.

For every code snippet the user shares:
1. Write a complete docstring with: one-line summary, extended description
   (if the function is non-trivial), Args, Returns, Raises, and Example.
2. Follow Google Python Style Guide conventions exactly.
3. Infer parameter types from usage if no type hints are present.
4. If the code has an obvious bug, note it in a "Note:" section of the
   docstring rather than fixing the code.

Output ONLY the docstring — no other commentary."""

TEST_CODE_1 = """import time
def retry(func, max_attempts=3, delay=1.0):
    for attempt in range(max_attempts):
        try:
            return func()
        except Exception as e:
            if attempt == max_attempts - 1:
                raise
            time.sleep(delay)"""

TEST_CODE_2 = """def parse_price(text):
    cleaned = text.replace("₹", "").replace(",", "").strip()
    if not cleaned:
        raise ValueError("empty price")
    return float(cleaned)"""

print(INSTRUCTIONS, TEST_CODE_1, TEST_CODE_2, sep="\n\n")

**Check it like a developer:** paste the docstring you got between the triple quotes below and run the cell. `help()` prints what your teammates will see in their editor.

In [ ]:
import time

def retry(func, max_attempts=3, delay=1.0):
    """PASTE THE DOCSTRING HERE"""
    for attempt in range(max_attempts):
        try:
            return func()
        except Exception as e:
            if attempt == max_attempts - 1:
                raise
            time.sleep(delay)

help(retry)

## Section 4 · Level up: a skill is a folder in your repo *(slide 15 · try it at home)*

Under the form, a skill is a **folder** with a `SKILL.md` file: a short header (`name`, `description`) and then the instructions, plus any reference files. This is the **Agent Skills** format that Claude, ChatGPT and Codex, and Cursor use; the Gemini app can **import** a `SKILL.md` and **export** a skill as a .zip.

**Steps**
1. Run the next cell: it writes `review-python-code/SKILL.md` + `banned_patterns.md`, checks the header against the format's rules, and zips it.
2. Download the .zip (the cell after it).
3. gemini.google.com → Customize → Skills → **Create** → **Upload** → pick the .zip → review → **Create**.
4. Look at the folder like an engineer: this is what a team would keep in Git and change through pull requests.

In [ ]:
import os, re, zipfile

FOLDER = "review-python-code"
SKILL_MD = """---
name: review-python-code
description: Reviews Python code for bugs, security issues and PEP 8 style, rates readability and suggests one refactor. Use when the user shares Python code or asks for a code review.
---

You are a senior Python code reviewer at a top-tier tech company.

For every code snippet shared:
1. Check for bugs and logical errors.
2. Flag security vulnerabilities (SQL injection, hardcoded secrets, unsafe deserialization).
3. Note PEP 8 style violations.
4. Rate readability on a scale of 1-5.
5. Suggest the single most impactful refactor.

Use this structure: Bugs, Security, Style, Readability (X/5), Top Refactor (before/after code).
Keep reviews under 300 words. Be direct. Always suggest type hints if they're missing.
Also check the code against banned_patterns.md.
"""
BANNED = """# Banned patterns (payments team)
- Bare `except:`. Always catch a specific exception.
- `print()` in service code. Use the logging module.
- `eval()` or `exec()` on any input.
- Hardcoded secrets, API keys or passwords. Read them from environment variables.
- Mutable default arguments, like `def add(item, items=[])`.
- Unused imports and unused variables.
"""

def check_header(skill_md, folder):
    """Check the SKILL.md header against the Agent Skills rules: name and description are required."""
    head = skill_md.split("---")[1]
    fields = dict(line.split(":", 1) for line in head.strip().splitlines())
    name, desc = fields.get("name", "").strip(), fields.get("description", "").strip()
    problems = []
    if not re.fullmatch(r"[a-z0-9]+(-[a-z0-9]+)*", name) or len(name) > 64:
        problems.append("name: lowercase letters, numbers and single hyphens, up to 64 characters")
    if name != folder:
        problems.append("name must match the folder name")
    if not desc or len(desc) > 1024:
        problems.append("description: required, up to 1,024 characters")
    return problems

assert check_header(SKILL_MD, FOLDER) == []
assert check_header(SKILL_MD.replace("review-python-code", "Review_Python"), FOLDER)  # a bad name is caught

os.makedirs(FOLDER, exist_ok=True)
with open(f"{FOLDER}/SKILL.md", "w", encoding="utf-8") as f:
    f.write(SKILL_MD)
with open(f"{FOLDER}/banned_patterns.md", "w", encoding="utf-8") as f:
    f.write(BANNED)
with zipfile.ZipFile(f"{FOLDER}.zip", "w") as z:   # SKILL.md sits in the zip's main folder, as Gemini's upload expects
    z.write(f"{FOLDER}/SKILL.md", "SKILL.md")
    z.write(f"{FOLDER}/banned_patterns.md", "banned_patterns.md")
print("✅ header OK ·", sorted(os.listdir(FOLDER)), "·", f"{FOLDER}.zip ready")

In [ ]:
try:
    from google.colab import files
    files.download("review-python-code.zip")
except ImportError:
    print("Not in Colab: review-python-code.zip is saved next to this notebook.")

## Section 5 · Hands-on: the Backend Engineering project *(slide 18 · 14 min)*

**Steps**
1. claude.ai → **Projects** (left panel) or claude.ai/projects → **+ New Project** → name `PROJECT_NAME`. (Claude doesn't read the name or description.)
2. **Set project instructions** → paste `PROJECT_INSTRUCTIONS` → **Save instructions**.
3. Run the two cells below: they create `api_style_guide.md` and download it. In the project, click **+** (right side) → upload it.
4. A new chat **inside the project**: send `CHAT_1`.
5. A **second** new chat inside the project: send `CHAT_2`.

**Watch for:** chat 1 uses SQLAlchemy, logging (not print), type hints and error handling with HTTP status codes, and you never asked. Chat 2 thinks in microservices without being told your stack.
**Post in Zoom chat:** one rule from `api_style_guide.md` you can see in your code.

**New in 2026:** each project also has **its own memory** (facts Claude remembers from that project's chats, kept apart from your other projects; the "Use account memory" toggle on the project page controls mixing). **Stretch:** upload a real Python file from your own project and ask: "Write tests for this module."

In [ ]:
PROJECT_NAME = "Backend Engineering"

PROJECT_INSTRUCTIONS = """You are a senior backend engineer specialising in Python, FastAPI, and PostgreSQL.

Standing Context
- We follow Google Python Style Guide for all code
- All endpoints must include error handling with proper HTTP status codes
- Use the logging module, never print statements
- Database queries use SQLAlchemy ORM, never raw SQL
- All functions must have type hints and docstrings

Default Output Behaviour
- For code: provide production-ready code with error handling
- For architecture questions: think in microservices, explain trade-offs
- For debugging: ask clarifying questions before suggesting fixes
- For code review: use the format from our style guide

Safety Rules
- Never suggest hardcoded credentials or secrets
- Always recommend environment variables for configuration
- Flag any SQL injection vulnerabilities immediately"""

CHAT_1 = "Write a FastAPI endpoint for user registration with email/password"
CHAT_2 = "How should we handle database connection pooling for our FastAPI app?"

print(PROJECT_INSTRUCTIONS, CHAT_1, CHAT_2, sep="\n\n")

In [ ]:
%%writefile api_style_guide.md
- All endpoints return JSON with keys: status, data, error
- Use plural nouns for collections: /users not /user
- Use snake_case for JSON fields
- Always include pagination for list endpoints
- Rate limiting headers required on all endpoints

In [ ]:
try:
    from google.colab import files
    files.download("api_style_guide.md")
except ImportError:
    print("Not in Colab: api_style_guide.md is saved next to this notebook.")

## Section 6 · Live demo: Exercise 5.1, one email at three temperatures *(slide 25 · you repeat it at home)*

Class 1 taught you what temperature does. Today's twist: **the newest models ignore it** (Gemini 4 and 3.6+ ignore temperature, top-p and top-k). So this runs on a model that still uses it: Rishav names it in class (Gemma 4 is the usual pick).

**Steps**
1. aistudio.google.com → **Playground** → pick the model Rishav names.
2. **Run settings** (tune icon, top right) → **System instructions** = `SYSTEM`.
3. Temperature **0.1** → send `PROMPT`. New chat, same again.
4. Temperature **1.2**, twice, a new chat each time. (0.7 too, if there's time.)

**Predict first:** will 1.2 write a better or worse email than 0.1? **Watch for:** which one changes between runs?

| Temperature | Run 1 in one word | Run 2 in one word | Would you send it? |
|---|---|---|---|
| 0.1 | | | |
| 0.7 | | | |
| 1.2 | | | |

In [ ]:
SYSTEM = "You are a helpful writing assistant. Keep your responses clear and concise."

PROMPT = """Write a short welcome email for a new employee joining a software team.
Their name is Priya and she's joining as a Backend Developer."""

print(SYSTEM, PROMPT, sep="\n\n")

**Optional, in code** (needs a free Gemini API key: aistudio.google.com → **Get API key** → in Colab, the **key icon** → add `GEMINI_API_KEY`). No key? It just says "skipped".

In [ ]:
import os

def get_key():
    """Your free Gemini API key from Colab Secrets (key icon on the left) or an environment variable."""
    try:
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return os.environ.get("GEMINI_API_KEY")

MODEL = "gemma-4-31b-it"        # free; still uses temperature
# MODEL = "gemini-4-hydrogen"   # free; ignores temperature. Try it and compare.

key = get_key()
if not key:
    print("No GEMINI_API_KEY found: skipped. That's fine, the live demo covers it.")
else:
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=key)
    for t in (0.1, 0.7, 1.2):
        # The instructions go inside the message, which works on every model (Gemma included).
        reply = client.models.generate_content(model=MODEL, contents=SYSTEM + "\n\n" + PROMPT,
                                               config=types.GenerateContentConfig(temperature=t))
        print(f"--- temperature {t} ---\n{reply.text}\n")

## Section 7 · Run along: structured output on a messy incident report *(slide 26 · 6 min)*

In Class 3 you *asked* for JSON in a system prompt and checked it with `json.loads`; sometimes it came back wrapped in ``` marks. Today the **platform enforces the shape**: you give it a JSON **schema**, and the reply must fit it. This is Quiz 3's exact scenario.

**Steps**
1. aistudio.google.com → **Playground** → model **Gemini 4 Hydrogen** (free).
2. **Run settings** → find the **structured output** setting (labels move around; ask in chat if you can't find it) → switch it on → paste `SCHEMA_JSON` (printed by the next cell).
3. Send `PROMPT`. Then run it **a second time**.
4. Copy the JSON reply into the **checker** cell below and run it. **Post ✅ or ❌ in Zoom chat.**
5. Click **Get code**: the same request in Python, with the schema in the config.

**Watch for:** same keys, valid JSON, both runs, with no "please return JSON" in the prompt. And `duration_minutes`: the model has to work it out from the messy times.

In [ ]:
import json

INCIDENT_REPORT = """ok so around 9:40 pm last night UPI payments started failing on checkout for like 30-35 mins??
payments-service was timing out talking to the bank gateway. Rahul thinks the new retry config
(deployed 9:15) made it worse - retries piled up. ~4,800 orders failed, support got flooded.
rolled back the config at 10:14, things recovered by 10:17. we need alerts on gateway latency,
and someone pls load-test retry changes before deploy. sev? prob SEV1 since checkout was down"""

PROMPT = "Extract the incident details from this report.\n\n" + INCIDENT_REPORT

SCHEMA = {
    "type": "object",
    "properties": {
        "service": {"type": "string", "description": "the service that failed"},
        "severity": {"type": "string", "enum": ["SEV1", "SEV2", "SEV3"]},
        "started_at": {"type": "string", "description": "when it started, 24-hour HH:MM"},
        "duration_minutes": {"type": "integer", "description": "from start to recovery"},
        "customer_impact": {"type": "string"},
        "root_cause": {"type": "string"},
        "action_items": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["service", "severity", "started_at", "duration_minutes",
                 "customer_impact", "root_cause", "action_items"],
}

SCHEMA_JSON = json.dumps(SCHEMA, indent=2)
print(PROMPT, SCHEMA_JSON, sep="\n\n")

**The checker** (runs offline). It tests the reply against the schema, like the code in a real app would before saving it to a database.

In [ ]:
import json

def check_incident(text):
    """Problems with a reply, checked against SCHEMA. An empty list means it passes."""
    try:
        data = json.loads(text)
    except json.JSONDecodeError as e:
        return [f"not valid JSON: {e.msg}"]
    if not isinstance(data, dict):
        return ["not a JSON object"]
    problems = [f'missing "{k}"' for k in SCHEMA["required"] if k not in data]
    py_type = {"string": str, "integer": int, "array": list}
    for key, spec in SCHEMA["properties"].items():
        if key not in data:
            continue
        value, want = data[key], py_type[spec["type"]]
        if not isinstance(value, want) or (want is int and isinstance(value, bool)):
            problems.append(f'"{key}" should be a {spec["type"]}')
        elif "enum" in spec and value not in spec["enum"]:
            problems.append(f'"{key}" must be one of {spec["enum"]}')
    return problems

# Self-check: a clean reply passes; the Class 3 problem (JSON wrapped in ``` marks) fails.
GOOD = ('{"service": "payments-service", "severity": "SEV1", "started_at": "21:40", "duration_minutes": 37, '
        '"customer_impact": "4,800 UPI orders failed", "root_cause": "retry config piled up timeouts", '
        '"action_items": ["alert on gateway latency"]}')
assert check_incident(GOOD) == []
assert check_incident("```json\n" + GOOD + "\n```")[0].startswith("not valid JSON")
assert check_incident(GOOD.replace('"SEV1"', '"critical"')) == ['"severity" must be one of [\'SEV1\', \'SEV2\', \'SEV3\']']
print("checker OK")

In [ ]:
# Paste the model's reply between the triple quotes, then run this cell.
reply = """
PASTE THE REPLY HERE
"""

if "PASTE THE REPLY HERE" in reply:
    print("Paste the JSON reply between the triple quotes first, then run again.")
else:
    problems = check_incident(reply.strip())
    print("✅ matches the schema" if not problems else "❌ " + "; ".join(problems))

**Optional, in code:** the request **Get code** gives you, for Gemini 4 Hydrogen: the schema goes in `response_json_schema`, and `thinking_level="low"` keeps it fast and cheap. Needs `GEMINI_API_KEY`; skips cleanly without one.

In [ ]:
import os

def get_key():
    """Your free Gemini API key from Colab Secrets (key icon on the left) or an environment variable."""
    try:
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return os.environ.get("GEMINI_API_KEY")

key = get_key()
if not key:
    print("No GEMINI_API_KEY found: skipped. The AI Studio run covers it.")
else:
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=key)
    response = client.models.generate_content(
        model="gemini-4-hydrogen",
        contents=PROMPT,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_json_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="low"),
        ),
    )
    print(response.text)
    print(check_incident(response.text) or "✅ matches the schema")

## Section 8 · Hands-on: mini build *(slide 30 · 10 min in class, finish at home)*

**First ask:** is this one job, or many jobs sharing the same context? That picks A or B.

### A · "The Landlord's Lifeline": Sam's Tenant Helper (skill, or Gem before 17 Nov)
Sam owns rental flats. Tenants keep asking about shelves, repairs and rent dates.
1. A new skill named `SAM_SKILL_NAME` (or a Gem), description `SAM_DESCRIPTION`, instructions `SAM_INSTRUCTIONS`.
2. File: run the `lease_summary.txt` cell below, download it, add it to the skill.
3. Test: `SAM_TEST`. **Bonus:** ask in a new chat without "/" and see if Gemini picks the skill.

**Success:** it answers from your file ("According to your lease agreement…") and quotes the real point. A generic answer means the instructions or the file need work.

### B · "The Thesis Whisperer": Meera's Thesis Lab (Claude Project)
Meera writes her master's thesis from research papers and her advisor's feedback.
1. New project. Instructions: `MEERA_INSTRUCTIONS` (put your own topic in the brackets).
2. Files: 2+ documents: real open-access papers from Google Scholar, **or** the two practice summaries below (clearly marked as made up).
3. Chat 1: `MEERA_CHAT_1`. Chat 2: `MEERA_CHAT_2`.

**Success:** the paragraph cites your uploaded papers, the summary matches the file, and APA style appears without reminding. *APA = a common format for citing papers.*

### C · "The Spice Dial" (AI Studio, optional)
Tomas's newsletter needs the best temperature for micro-stories. System instructions `TOMAS_SYSTEM`, prompt `TOMAS_PROMPT`, at **0.2, 0.6, 1.0**, on a model whose temperature works.
**Success:** you can say in 1–2 sentences what changed and which temperature you'd pick, and why.

### Or your own
Your skill candidate from slide 9: a commit-message writer, a standup summariser, a SQL explainer, a PR-description writer…

**Post in Zoom chat:** your assistant's name + one test question.

In [ ]:
SAM_SKILL_NAME = "answer-tenant-questions"
SAM_DESCRIPTION = ("Answers tenants' questions about Sam's rental flats from the lease summary. "
                   "Use when a tenant asks about rent, repairs, pets, guests or changes to the flat.")

SAM_INSTRUCTIONS = """You are a property management assistant for Sam's rental flats.
Tone: friendly but professional.
Answer tenants' questions using ONLY the lease summary file.
Start answers with "According to your lease agreement..." and quote the relevant point.
Never give legal advice. If the lease doesn't cover the question, or it is unclear,
say so and ask the tenant to contact Sam directly."""

SAM_TEST = "Can I paint my bedroom walls dark blue?"

print(SAM_DESCRIPTION, SAM_INSTRUCTIONS, SAM_TEST, sep="\n\n")

In [ ]:
%%writefile lease_summary.txt
Lease Agreement Summary: Flat 3B, Green Park Residency (owner: Sam)
1. Rent: ₹18,000 a month, due by the 5th. Late fee ₹500 after the 7th.
2. Security deposit: two months' rent, returned within 30 days of moving out.
3. Small repairs under ₹1,000: the tenant pays. Plumbing, wiring and supplied appliances: Sam pays.
4. Report repairs to Sam on WhatsApp, with a photo.
5. Pets: small pets only, with Sam's written approval. No pets in common areas.
6. Quiet hours: 10 pm to 7 am.
7. No painting, drilling or wall shelves without Sam's written permission.
8. Removable hooks and stick-on fixtures are allowed.
9. Guests may stay up to 7 nights in a row; longer needs approval.
10. Notice period: one month from either side.
11. Society rules apply: parking, lift use, garbage timings.
12. Subletting is not allowed.

In [ ]:
MEERA_INSTRUCTIONS = """You are an academic writing partner for my master's thesis on [YOUR THESIS TOPIC].
Use APA 7th edition for every citation.
Every claim must cite one of the papers uploaded to this project.
If the uploaded papers don't support a claim, say so instead of inventing a source."""

MEERA_CHAT_1 = "Write one Literature Review paragraph for my thesis, using the uploaded papers."
MEERA_CHAT_2 = "Summarise the first uploaded paper in 5 bullet points."

TOMAS_SYSTEM = """Micro-fiction writer. Stories under 300 words, clear beginning-middle-end,
literary fiction, unexpected but logical twist ending."""

TOMAS_PROMPT = """Write a micro-story about a librarian who discovers that one book
in the collection has a different ending every time someone reads it."""

print(MEERA_INSTRUCTIONS, MEERA_CHAT_1, MEERA_CHAT_2, TOMAS_SYSTEM, TOMAS_PROMPT, sep="\n\n")

**Practice papers for B** (made up, for practice only; use real papers if you have them). Run both cells, then the download cell.

In [ ]:
%%writefile paper1_summary.txt
PRACTICE SUMMARY (made up for this class, not a real paper)
Title: AI review comments on student pull requests
Authors: A. Rao and K. Mehta (practice example)
Question: Do students accept code-review comments written by an AI assistant?
Method: 40 students, 600 pull requests over one semester; half got AI comments first, half got only human comments.
Findings: students accepted 61% of AI comments and 74% of human comments. AI comments on naming and style were accepted most; comments on design were accepted least.
Limits: one university, one semester, small projects only.

In [ ]:
%%writefile paper2_summary.txt
PRACTICE SUMMARY (made up for this class, not a real paper)
Title: Saved instructions and consistency of AI coding help
Authors: S. Iyer (practice example)
Question: Do saved instructions (like a skill or a Project) make AI answers more consistent?
Method: the same 50 coding questions asked 3 times each, with and without saved team rules.
Findings: with saved rules, answers followed the team style in 88% of runs, without them in 41%. Saved rules did not make answers more correct, only more consistent.
Limits: one model, one programming language (Python).

In [ ]:
import os
try:
    from google.colab import files
    for name in ("lease_summary.txt", "paper1_summary.txt", "paper2_summary.txt"):
        if os.path.exists(name):
            files.download(name)
except ImportError:
    print("Not in Colab: the files are saved next to this notebook.")

## Section 9 · Homework and links

1. **Finish your mini build** (A, B, C or your own).
2. **Upload the skill folder** from Section 4 (Customize → Skills → Create → Upload).
3. **Temperature test** at home, on the model Rishav named (Section 6).
4. **Exercise 5.2 (optional): multi-turn testing.** AI Studio, same `SYSTEM` as Section 6, temperature **0.5** on a model that uses it. Send `TURN_1`, then `TURN_2`, then `TURN_3`, all in **one** chat.
   - Does it remember the budget (₹50,000), the team size (8) and the place (Bangalore) from turn 1 without being told again?
   - Does turn 2 add to the plan or start over? Does the turn-3 table include everything?
   - If it forgets: that's the context window from Class 1.
5. **Class 4 assignment and homework** on the Scaler dashboard. **Read the post-read.** Keep your skill and your project: Class 5 explains how they search your files.

**Why saved context is cheap (for the curious):** providers cache the start of a request that repeats. Gemini 4 Hydrogen bills cached input at $0.075 per million tokens instead of $1.50; Claude Sonnet 5.5 at $0.10 instead of $2 (vendor pricing pages, 13 Nov 2026). So put long, fixed instructions first.

| What | Link |
|---|---|
| Pre-read | https://drive.google.com/open?id=1kwg5DX275_Jch_jkoeRsFktZy1EAF5l8yhLaZgjOFYU |
| Post-read | https://drive.google.com/open?id=1nWuDf-US3fIwSFwwvRKZTEo8mGMfqD0r |
| Assignment | https://drive.google.com/file/d/1MsNXdjt2ZFDo8U9X0ZT9o3V9_XhWy5Pi/view?usp=sharing |
| Tools | gemini.google.com · claude.ai/projects · aistudio.google.com |

In [ ]:
TURN_1 = """Help me plan a team offsite for 8 engineers. We have a budget
of ₹50,000 and it should be a one-day event near Bangalore."""

TURN_2 = """Good ideas. Now add a 30-minute team-building activity to the
schedule. Something fun that engineers would actually enjoy."""

TURN_3 = """Can you put the whole plan into a clean table with columns:
Time, Activity, Location, and Estimated Cost?"""

print(TURN_1, TURN_2, TURN_3, sep="\n\n")